# MobileNetV2 Image Classifier

Classify an image with TensorFlow's ImageNet-pretrained MobileNetV2. This notebook can run locally or in Google Colab.

## Install dependencies (if needed)

Uncomment the next cell in a fresh Jupyter environment. In Colab, TensorFlow, NumPy, and Matplotlib are usually preinstalled.

In [ ]:
# %pip install tensorflow numpy matplotlib pillow

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

print('TensorFlow:', tf.__version__)

## Load the pretrained model

The first run may download the ImageNet weights. The classification head is included, so the model predicts ImageNet classes directly.

In [ ]:
model = tf.keras.applications.MobileNetV2(
    weights='imagenet',
    include_top=True,
    input_shape=(224, 224, 3),
)
model.summary()

## Choose an image

In Colab, upload a file. In local Jupyter, enter the image path (or set `IMAGE_PATH` directly).

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    image_path = next(iter(uploaded))
except ImportError:
    image_path = input('Enter the path to an image file: ').strip()

if not Path(image_path).is_file():
    raise FileNotFoundError(f'Image not found: {image_path}')

img = Image.open(image_path).convert('RGB').resize((224, 224))
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## Preprocess and classify

MobileNetV2 preprocessing scales the pixel values as expected by the pretrained model. Add a batch dimension before prediction.

In [ ]:
img_array = np.asarray(img, dtype=np.float32)
img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
img_array_expanded = np.expand_dims(img_array, axis=0)
print('Model input shape:', img_array_expanded.shape)

predictions = model.predict(img_array_expanded, verbose=0)
top_predictions = tf.keras.applications.mobilenet_v2.decode_predictions(predictions, top=5)[0]

for rank, (_, label, score) in enumerate(top_predictions, start=1):
    print(f'{rank}: {label.replace("_", " " )} ({score:.2%})')

In [ ]:
best_label = top_predictions[0][1].replace('_', ' ').title()
confidence = top_predictions[0][2] * 100
plt.figure(figsize=(6, 5))
plt.imshow(img)
plt.axis('off')
plt.title(f'Prediction: {best_label}\nConfidence: {confidence:.2f}%')
plt.show()

## About the result

This is an ImageNet classification demo with a pretrained model; it has not been trained or fine-tuned on a custom dataset. Confidence scores can be wrong, especially for image types outside ImageNet's classes.